In [1]:
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

# load
data_path = '../input/playground-series-s5e6/'
train = pd.read_csv(data_path + 'train.csv')
test  = pd.read_csv(data_path + 'test.csv')
print(f"Train {train.shape}, Test {test.shape}")

Train (750000, 10), Test (250000, 9)


In [2]:
# 2. Simple Feature Engineering (fixed names)

# 2.1 Specify your macronutrient columns explicitly
macro_cols = ['Nitrogen', 'Phosphorous', 'Potassium']
print("Using macros:", macro_cols)

# 2.2 Create pairwise ratios
for df in (train, test):
    for a in macro_cols:
        for b in macro_cols:
            if a == b:
                continue
            df[f"{a}_{b}_ratio"] = df[a] / (df[b] + 1e-5)

# 2.3 (Optional) log1p transforms
to_log = macro_cols + [f"{a}_{b}_ratio" 
                       for a in macro_cols 
                       for b in macro_cols if a != b]
for col in to_log:
    for df in (train, test):
        df[f"log1p_{col}"] = np.log1p(df[col])

# 2.4 Verify
print("New train shape:", train.shape)
print("New test  shape:",  test.shape)

Using macros: ['Nitrogen', 'Phosphorous', 'Potassium']
New train shape: (750000, 25)
New test  shape: (250000, 24)


In [3]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 3.1 Encode the target fertilizer names in train
le = LabelEncoder()
train['target'] = le.fit_transform(train['Fertilizer Name'])

# 3.2 Encode Crop Type on both sets
crop_le = LabelEncoder()
train['Crop_Type_enc'] = crop_le.fit_transform(train['Crop Type'])
test ['Crop_Type_enc'] = crop_le.transform(test ['Crop Type'])

# 3.3 Define drop lists
train_drop = ['id', 'Fertilizer Name', 'Crop Type', 'target']
test_drop  = ['id', 'Crop Type']

# 3.4 Build feature matrices
X_train = train.drop(columns=train_drop)
y_train = train['target']
X_test  = test .drop(columns=test_drop)

# 3.5 Sanity check prints
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("Number of target classes:", len(le.classes_))

X_train shape: (750000, 23)
X_test  shape: (250000, 23)
Number of target classes: 7


In [4]:
from sklearn.preprocessing import LabelEncoder

# Encode Soil Type
soil_le = LabelEncoder()
train['Soil_Type_enc'] = soil_le.fit_transform(train['Soil Type'])
test ['Soil_Type_enc'] = soil_le.transform(test ['Soil Type'])

# Now rebuild X_train & X_test to drop the text columns
train_drop = ['id', 'Fertilizer Name', 'Crop Type', 'Soil Type', 'target']
test_drop  = ['id', 'Crop Type', 'Soil Type']

X_train = train.drop(columns=train_drop)
y_train = train['target']
X_test  = test .drop(columns=test_drop)

# Sanity check
print("X_train columns now include:", X_train.columns.tolist()[:10], "…")
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

X_train columns now include: ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous', 'Nitrogen_Phosphorous_ratio', 'Nitrogen_Potassium_ratio', 'Phosphorous_Nitrogen_ratio', 'Phosphorous_Potassium_ratio'] …
X_train shape: (750000, 23)
X_test  shape: (250000, 23)


In [5]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import numpy as np

# 4.1 MAP@3 function
def map3_score(y_true, y_proba):
    top3 = np.argsort(y_proba, axis=1)[:, -3:][:, ::-1]
    score = 0.0
    for row, true in zip(top3, y_true):
        if true in row:
            score += 1.0 / (list(row).index(true) + 1)
    return score / len(y_true)

# 4.2 Prep
features = X_train.columns.tolist()
num_classes = len(le.classes_)
oof_preds = np.zeros((len(X_train), num_classes))
test_preds = np.zeros((len(X_test),  num_classes))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=['Crop_Type_enc'])
    dvalid = lgb.Dataset(X_val, label=y_val, categorical_feature=['Crop_Type_enc'])

    params = {
        'objective': 'multiclass',
        'num_class': num_classes,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'seed': 42,
    }

    clf = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )

    # Gather preds
    oof_preds[val_idx] = clf.predict(X_val)
    test_preds += clf.predict(X_test) / skf.n_splits

    print(f"Fold {fold} MAP@3:", map3_score(y_val, oof_preds[val_idx]))

# Final OOF score
print("Overall OOF MAP@3:", map3_score(y_train, oof_preds))

Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 1.92844
[200]	valid_0's multi_logloss: 1.92538
[300]	valid_0's multi_logloss: 1.92393
[400]	valid_0's multi_logloss: 1.92288
[500]	valid_0's multi_logloss: 1.92223
[600]	valid_0's multi_logloss: 1.92171
[700]	valid_0's multi_logloss: 1.92138
[800]	valid_0's multi_logloss: 1.92115
[900]	valid_0's multi_logloss: 1.92098
[1000]	valid_0's multi_logloss: 1.92092
Did not meet early stopping. Best iteration is:
[991]	valid_0's multi_logloss: 1.9209
Fold 1 MAP@3: 0.32855222222229585
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 1.92855
[200]	valid_0's multi_logloss: 1.9253
[300]	valid_0's multi_logloss: 1.92348
[400]	valid_0's multi_logloss: 1.92246
[500]	valid_0's multi_logloss: 1.92177
[600]	valid_0's multi_logloss: 1.92125
[700]	valid_0's multi_logloss: 1.9207
[800]	valid_0's multi_logloss: 1.92046
[900]	valid_0's multi_logloss: 1.92016
[1000]	valid_0's m

In [6]:
# 5. Build submission from test_preds (LightGBM)
import numpy as np

# 5.1 For each test row, get top-3 class indices
top3_ints = np.argsort(test_preds, axis=1)[:, -3:][:, ::-1]

# 5.2 Flatten, inverse-transform, then reshape into (n_rows, 3)
flat = top3_ints.ravel()
flat_labels = le.inverse_transform(flat)
top3_names = flat_labels.reshape(top3_ints.shape)

# 5.3 Build and save DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'Fertilizer Name': [' '.join(row) for row in top3_names]
})
submission.to_csv('submission.csv', index=False)
print(submission.head())


       id          Fertilizer Name
0  750000       DAP 28-28 14-35-14
1  750001  17-17-17 20-20 10-26-26
2  750002     20-20 10-26-26 28-28
3  750003   14-35-14 17-17-17 Urea
4  750004     20-20 10-26-26 28-28
